#### 导入包

In [12]:
# 导入所有必要的库
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import requests
from PIL import Image
# import cv2
import os
import re
from sklearn.model_selection import train_test_split
# 用于文本处理
from emoji import demojize
import opencc
# 用于CLIP分词 (这里以transformers库中的中文CLIP为例)
from transformers import ChineseCLIPProcessor
# 用于图像变换
import torchvision.transforms as transforms
# 设置随机种子保证可复现
torch.manual_seed(42)
np.random.seed(42)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.0 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/opt/conda/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/opt/conda/lib/python3.9/runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "/opt/conda/lib/python3.9/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/opt/conda/lib/python3.9/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/opt/conda/lib/python3.9/site-packages/ipykernel/kern

AttributeError: _ARRAY_API not found

ImportError: numpy.core._multiarray_umath failed to import

ImportError: numpy.core.umath failed to import

In [18]:
import torch

# 1. 检查CUDA是否可用
is_available = torch.cuda.is_available()
print(f"CUDA is available: {is_available}")

if is_available:
    # 2. 获取GPU的数量
    gpu_count = torch.cuda.device_count()
    print(f"Number of GPUs available: {gpu_count}")

    # 3. 获取当前GPU的名称
    current_gpu_name = torch.cuda.get_device_name(0)
    print(f"Name of the current GPU: {current_gpu_name}")

    # 4. 获取当前GPU的设备ID
    current_device = torch.cuda.current_device()
    print(f"ID of the current GPU: {current_device}")
else:
    print("PyTorch was installed with GPU support, but no compatible NVIDIA GPU/driver was found.")

CUDA is available: False
PyTorch was installed with GPU support, but no compatible NVIDIA GPU/driver was found.


#### 初步分析数据

In [19]:
# 加载数据
df_raw = pd.read_excel('/mnt/workspace/oss/yyj_ai/fake_reviews_detection/data/reviews_ori.xlsx')
print(f"数据集形状: {df_raw.shape}")
print("\n前5行数据:")
display(df_raw.head())
print("\n列信息:")
print(df_raw.dtypes)
print("\n标签分布:")
print(df_raw['label'].value_counts())

数据集形状: (7844, 6)

前5行数据:


,user_id,review_time,image_url,review_text,label,genre
0,花***灵,2021-06-08,https://jvod.300hu.com/img/2021/81489379/1/img...,JM家面膜，多年来一直在用。这款水母面膜，适合各种肤质，里面的精华很多，补水效果效果很明显呢...,1,美妆
1,不***头,2022-12-24,https://img30.360buyimg.com/shaidan/s300x300_j...,挑了好久，终于选择这款JM的水库面膜，简直是太棒了，强效水光，一剂深入补水，轻松应急，每一片...,1,美妆
2,新***络,2022-07-20,https://img30.360buyimg.com/shaidan/s300x300_j...,一直在用，JM 的水库面膜，简直是太棒了，双十一活动，我就多屯了一些朋友和我都觉得特别好用，...,1,美妆
3,L***家,2023-07-21,https://img30.360buyimg.com/shaidan/s300x300_j...,一直在用新款面膜，效果很赞！特别是三部曲，三步护理，洗面奶，眼霜，精华，感觉竟然比自己买的某...,1,美妆
4,j***3,2022-06-17,https://img30.360buyimg.com/shaidan/s300x300_j...,JM 的面膜，简直是太棒了，朋友和我都觉得特别好用， 我觉得各种肌肤都挺实用，尤其是那种长期...,1,美妆



列信息:
user_id        object
review_time    object
image_url      object
review_text    object
label           int64
genre          object
dtype: object

标签分布:
label
0    4221
1    3623
Name: count, dtype: int64


### 文本清洗

本文在文本预处理阶段采用轻量级清洗策略，仅去除 URL、平台噪声标签、异常符号，并保留评论的原始语言结构与情感表达。这是因为 CLIP 模型的文本编码器基于 Transformer 架构，能够自动建模上下文语义关系，过度分词或人工删除词汇反而会破坏跨模态对齐效果。实验中发现，大多数评论在清洗前后语义保持一致，验证了该策略在信息保留与噪声抑制之间的合理平衡。

In [ ]:
# 手动下载bert-base-chinese后测试是否可以使用

from transformers import BertModel, BertTokenizer

# 这里手动下载模型与分词器，根据目录加载使用

vocab_file = '/mnt/workspace/oss/yyj_ai/fake_reviews_detection/model/vocab.txt'
tokenizer = BertTokenizer(vocab_file)
bert = BertModel.from_pretrained("/mnt/workspace/oss/yyj_ai/fake_reviews_detection/model/bert-base-chinese/")

# 这里只是用到分词器，上一句调用模型不需要亦可
sents = ["白日依山尽", "黄河入海流"]
out = tokenizer.encode(
    # 传入的两个句子
    text=sents[0],
    text_pair=sents[1],
    # 长度大于设置是否截断
    truncation=True,
    # 一律补齐，如果长度不够
    padding='max_length',
    add_special_tokens=True,
    max_length=30,
    return_tensors=None,
)
print(out)
print(tokenizer.decode(out))

[101, 4635, 3189, 898, 2255, 2226, 102, 7942, 3777, 1057, 3862, 3837, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[CLS] 白 日 依 山 尽 [SEP] 黄 河 入 海 流 [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]


In [ ]:
import pandas as pd
import re
import opencc
from transformers import BertTokenizer

# 初始化繁简转换器
cc = opencc.OpenCC('t2s')

# 文本清洗函数
SHORT_THR = 5

def clean_text_for_clip(text):
    """清洗单条评论文本，适配中文CLIP文本编码"""
    if pd.isna(text):
        return None
    s = str(text)
    s = cc.convert(s)                         # 繁体转简体
    s = re.sub(r'https?://\S+|www\.\S+', ' ', s)  # 移除网址
    s = re.sub(r'@\w+|#\w+#?', ' ', s)           # 移除@和话题标签
    s = re.sub(r'【.*?】', ' ', s)                # 移除平台标签
    s = re.sub(r'[^\u4e00-\u9fa5a-zA-Z0-9\s.,!?，。！？]', ' ', s)  # 保留中英文数字和常用标点
    s = re.sub(r'\s+', ' ', s).strip()         # 合并多余空白字符
    return s if len(s) >= SHORT_THR else None

# 加载中文BERT tokenizer（Chinese CLIP文本编码器就是它）
vocab_file = '/mnt/workspace/oss/yyj_ai/fake_reviews_detection/model/vocab.txt'
tokenizer = BertTokenizer(vocab_file)
bert = BertModel.from_pretrained("/mnt/workspace/oss/yyj_ai/fake_reviews_detection/model/bert-base-chinese/")
# 测试
# print(tokenizer.tokenize("这件衣服质量太差了！"))

['这', '件', '衣', '服', '质', '量', '太', '差', '了', '！']


In [ ]:
def tokenize_text(text):
    """使用BERT tokenizer对文本进行分词和编码"""
    inputs = tokenizer(
        text,
        padding='max_length',
        truncation=True,
        max_length=77,
        return_tensors="pt"
    )
    return inputs['input_ids'].squeeze(0)  # 返回形状为 [77] 的Tensor

# 应用清洗
df_raw['text_clean'] = df_raw['review_text'].apply(clean_text_for_clip)
df_raw = df_raw.dropna(subset=['text_clean']).reset_index(drop=True)
print(f"清洗后剩余样本数: {len(df_raw)}")

# 测试
for i in range(10):
    print(f"原文: {df_raw.loc[i, 'review_text']}")
    print(f"清洗后: {df_raw.loc[i, 'text_clean']}")
    tokens = tokenize_text(df_raw.loc[i, 'text_clean'])
    print(f"Token IDs shape: {tokens.shape}\n")

清洗后剩余样本数: 7802
原文: JM家面膜，多年来一直在用。这款水母面膜，适合各种肤质，里面的精华很多，补水效果效果很明显呢！面膜的贴合度也不错，我是敷二十分钟取下，轻拍肌肤后再⬆️护肤品，第二天肌肤水汪汪呢，好上妆，持久一整天哟！
清洗后: JM家面膜，多年来一直在用。这款水母面膜，适合各种肤质，里面的精华很多，补水效果效果很明显呢！面膜的贴合度也不错，我是敷二十分钟取下，轻拍肌肤后再 护肤品，第二天肌肤水汪汪呢，好上妆，持久一整天哟！
Token IDs shape: torch.Size([77])

原文: 挑了好久，终于选择这款JM的水库面膜，简直是太棒了，强效水光，一剂深入补水，轻松应急，每一片面膜富含35g水润精华。这次趁着双十一活动,我就多屯了一些朋友和我都觉得特别好用，这款面膜特别适合干皮混干皮油皮，我觉得各种肌肤都挺实用，尤其是那种长期缺水的用于急救，特别管用。第一次使用连续三天都连着贴，之后一周使用两次就可以，用完皮肤滑滑的嫩嫩的，简直像皮肤喝饱了水一样
清洗后: 挑了好久，终于选择这款JM的水库面膜，简直是太棒了，强效水光，一剂深入补水，轻松应急，每一片面膜富含35g水润精华。这次趁着双十一活动,我就多屯了一些朋友和我都觉得特别好用，这款面膜特别适合干皮混干皮油皮，我觉得各种肌肤都挺实用，尤其是那种长期缺水的用于急救，特别管用。第一次使用连续三天都连着贴，之后一周使用两次就可以，用完皮肤滑滑的嫩嫩的，简直像皮肤喝饱了水一样
Token IDs shape: torch.Size([77])

原文: 一直在用，JM 的水库面膜，简直是太棒了，双十一活动，我就多屯了一些朋友和我都觉得特别好用， JM 这款面膜特别适合干皮混干皮油皮，我觉得各种肌肤都挺实用，尤其是那种长期缺水的用于急救，特别管用。第一次使用连续三天都连着贴，之后一周使用两次就可以，用完皮肤滑滑的嫩嫩的，简直像皮肤喝饱了水一样。
清洗后: 一直在用，JM 的水库面膜，简直是太棒了，双十一活动，我就多屯了一些朋友和我都觉得特别好用， JM 这款面膜特别适合干皮混干皮油皮，我觉得各种肌肤都挺实用，尤其是那种长期缺水的用于急救，特别管用。第一次使用连续三天都连着贴，之后一周使用两次就可以，用完皮肤滑滑的嫩嫩的，简直像皮肤喝饱了水一样。
Token IDs shape: torch.

In [35]:
df_raw

,user_id,review_time,image_url,review_text,label,genre,text_clean
0,花***灵,2021-06-08,https://jvod.300hu.com/img/2021/81489379/1/img...,JM家面膜，多年来一直在用。这款水母面膜，适合各种肤质，里面的精华很多，补水效果效果很明显呢...,1,美妆,JM家面膜，多年来一直在用。这款水母面膜，适合各种肤质，里面的精华很多，补水效果效果很明显呢...
1,不***头,2022-12-24,https://img30.360buyimg.com/shaidan/s300x300_j...,挑了好久，终于选择这款JM的水库面膜，简直是太棒了，强效水光，一剂深入补水，轻松应急，每一片...,1,美妆,挑了好久，终于选择这款JM的水库面膜，简直是太棒了，强效水光，一剂深入补水，轻松应急，每一片...
2,新***络,2022-07-20,https://img30.360buyimg.com/shaidan/s300x300_j...,一直在用，JM 的水库面膜，简直是太棒了，双十一活动，我就多屯了一些朋友和我都觉得特别好用，...,1,美妆,一直在用，JM 的水库面膜，简直是太棒了，双十一活动，我就多屯了一些朋友和我都觉得特别好用，...
3,L***家,2023-07-21,https://img30.360buyimg.com/shaidan/s300x300_j...,一直在用新款面膜，效果很赞！特别是三部曲，三步护理，洗面奶，眼霜，精华，感觉竟然比自己买的某...,1,美妆,一直在用新款面膜，效果很赞！特别是三部曲，三步护理，洗面奶，眼霜，精华，感觉竟然比自己买的某...
4,j***3,2022-06-17,https://img30.360buyimg.com/shaidan/s300x300_j...,JM 的面膜，简直是太棒了，朋友和我都觉得特别好用， 我觉得各种肌肤都挺实用，尤其是那种长期...,1,美妆,JM 的面膜，简直是太棒了，朋友和我都觉得特别好用， 我觉得各种肌肤都挺实用，尤其是那种长期...
...,...,...,...,...,...,...,...
7797,匿名买家,2025年6月22日,//img.alicdn.com/imgextra/i3/46116860184273839...,人都要疯了 用完狂爆痘！！！真的要谨慎购买,0,美妆,人都要疯了 用完狂爆痘！！！真的要谨慎购买
7798,匿名买家,2025年6月21日,//img.alicdn.com/imgextra/i2/46116860184273847...,没有塑封膜，之前买过多次50毫升和15毫升都是有塑封膜，这次没有，怀疑是人家退货的，店铺要求...,0,美妆,没有塑封膜，之前买过多次50毫升和15毫升都是有塑封膜，这次没有，怀疑是人家退货的，店铺要求...
7799,微***,2025年6月20日,//img.alicdn.com/imgextra/i2/46116860184273848...,之前一直用兰蔻，这次想换换。就买了黑绷带和sk2的大红瓶面霜试试。官网应该不至于卖假货但收到...,0,美妆,之前一直用兰蔻，这次想换换。就买了黑绷带和sk2的大红瓶面霜试试。官网应该不至于卖假货但收到...
7800,匿名买家,2025年6月20日,//img.alicdn.com/imgextra/i4/46116860184273824...,几千块的护肤品，让一起发个礼袋，说了急要送人，还要确认收货才能送，送过来的这个礼袋什么玩意，...,0,美妆,几千块的护肤品，让一起发个礼袋，说了急要送人，还要确认收货才能送，送过来的这个礼袋什么玩意，...


### 图像清洗

对于大型数据集，不要一次性下载所有图片。可以边下载边处理，或使用缓存机制。
考虑使用DataLoader的num_workers参数进行并行数据加载，但注意不要在Notebook中设置过高（通常2-4为宜）。

In [46]:
pip install openai-clip

Looking in indexes: https://mirrors.aliyun.com/pypi/simple/
Note: you may need to restart the kernel to use updated packages.


In [47]:
# 检查
import clip
print(clip.__file__)

/opt/conda/envs/py3.11/lib/python3.11/site-packages/clip/__init__.py


In [50]:
# 图片下载模块
import os
import requests
import pandas as pd
from tqdm import tqdm
import hashlib

IMG_SAVE_DIR = "/mnt/workspace/oss/yyj_ai/fake_reviews_detection/images"
os.makedirs(IMG_SAVE_DIR, exist_ok=True)

HEADERS_LIST = [
    {
        "User-Agent": "Mozilla/5.0",
        "Referer": "https://www.tmall.com/"
    },
    {
        "User-Agent": "Mozilla/5.0",
        "Referer": "https://detail.tmall.com/"
    },
    {
        "User-Agent": "Mozilla/5.0",
        "Referer": "https://www.jd.com/"
    },
    {"User-Agent": "Mozilla/5.0"}
]


def normalize_url(url: str) -> str:
    """处理 // 开头的 CDN URL"""
    url = url.strip()
    if url.startswith("//"):
        return "https:" + url
    return url

def infer_ext_from_content_type(content_type: str) -> str:
    """从 HTTP Header 判断真实图片格式"""
    content_type = content_type.lower()
    if "jpeg" in content_type or "jpg" in content_type:
        return ".jpg"
    if "png" in content_type:
        return ".png"
    if "webp" in content_type:
        return ".webp"
    return ".jpg"

def safe_filename(url: str, ext: str) -> str:
    """URL → 稳定文件名（避免重复）"""
    return hashlib.md5(url.encode("utf-8")).hexdigest() + ext


def download_image(url: str) -> str | None:
    """下载单张图片，成功返回本地路径"""
    if pd.isna(url) or not isinstance(url, str):
        return None

    url = normalize_url(url)

    for headers in HEADERS_LIST:
        try:
            r = requests.get(
                url,
                headers=headers,
                timeout=12,
                stream=True
            )

            if r.status_code != 200:
                continue

            content_type = r.headers.get("Content-Type", "")
            if "image" not in content_type.lower():
                continue

            ext = infer_ext_from_content_type(content_type)
            filename = safe_filename(url, ext)
            save_path = os.path.join(IMG_SAVE_DIR, filename)

            if os.path.exists(save_path):
                return save_path

            with open(save_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)

            return save_path

        except Exception:
            continue

    return None


def batch_download_images(df, url_col="image_url"):
    local_paths = []
    for url in tqdm(df[url_col], desc="Downloading images"):
        local_paths.append(download_image(url))
    return local_paths

In [51]:
# 批量下载并写回 DataFrame（一一对应）
df_raw = df_raw.copy()

df_raw["img_local_path"] = batch_download_images(df_raw)

success_rate = df_raw["img_local_path"].notna().mean()
print(f"图片下载成功率: {success_rate:.2%}")
print(f"成功下载数量: {df_raw['img_local_path'].notna().sum()}")

图片下载成功率: 99.85%
成功下载数量: 7790


In [52]:
import torch

# CLIP image: [3, 224, 224]
EMPTY_IMAGE = torch.zeros(3, 224, 224)

In [59]:
df_raw

,user_id,review_time,image_url,review_text,label,genre,text_clean,img_local_path
0,花***灵,2021-06-08,https://jvod.300hu.com/img/2021/81489379/1/img...,JM家面膜，多年来一直在用。这款水母面膜，适合各种肤质，里面的精华很多，补水效果效果很明显呢...,1,美妆,JM家面膜，多年来一直在用。这款水母面膜，适合各种肤质，里面的精华很多，补水效果效果很明显呢...,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
1,不***头,2022-12-24,https://img30.360buyimg.com/shaidan/s300x300_j...,挑了好久，终于选择这款JM的水库面膜，简直是太棒了，强效水光，一剂深入补水，轻松应急，每一片...,1,美妆,挑了好久，终于选择这款JM的水库面膜，简直是太棒了，强效水光，一剂深入补水，轻松应急，每一片...,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
2,新***络,2022-07-20,https://img30.360buyimg.com/shaidan/s300x300_j...,一直在用，JM 的水库面膜，简直是太棒了，双十一活动，我就多屯了一些朋友和我都觉得特别好用，...,1,美妆,一直在用，JM 的水库面膜，简直是太棒了，双十一活动，我就多屯了一些朋友和我都觉得特别好用，...,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
3,L***家,2023-07-21,https://img30.360buyimg.com/shaidan/s300x300_j...,一直在用新款面膜，效果很赞！特别是三部曲，三步护理，洗面奶，眼霜，精华，感觉竟然比自己买的某...,1,美妆,一直在用新款面膜，效果很赞！特别是三部曲，三步护理，洗面奶，眼霜，精华，感觉竟然比自己买的某...,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
4,j***3,2022-06-17,https://img30.360buyimg.com/shaidan/s300x300_j...,JM 的面膜，简直是太棒了，朋友和我都觉得特别好用， 我觉得各种肌肤都挺实用，尤其是那种长期...,1,美妆,JM 的面膜，简直是太棒了，朋友和我都觉得特别好用， 我觉得各种肌肤都挺实用，尤其是那种长期...,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
...,...,...,...,...,...,...,...,...
7797,匿名买家,2025年6月22日,//img.alicdn.com/imgextra/i3/46116860184273839...,人都要疯了 用完狂爆痘！！！真的要谨慎购买,0,美妆,人都要疯了 用完狂爆痘！！！真的要谨慎购买,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
7798,匿名买家,2025年6月21日,//img.alicdn.com/imgextra/i2/46116860184273847...,没有塑封膜，之前买过多次50毫升和15毫升都是有塑封膜，这次没有，怀疑是人家退货的，店铺要求...,0,美妆,没有塑封膜，之前买过多次50毫升和15毫升都是有塑封膜，这次没有，怀疑是人家退货的，店铺要求...,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
7799,微***,2025年6月20日,//img.alicdn.com/imgextra/i2/46116860184273848...,之前一直用兰蔻，这次想换换。就买了黑绷带和sk2的大红瓶面霜试试。官网应该不至于卖假货但收到...,0,美妆,之前一直用兰蔻，这次想换换。就买了黑绷带和sk2的大红瓶面霜试试。官网应该不至于卖假货但收到...,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
7800,匿名买家,2025年6月20日,//img.alicdn.com/imgextra/i4/46116860184273824...,几千块的护肤品，让一起发个礼袋，说了急要送人，还要确认收货才能送，送过来的这个礼袋什么玩意，...,0,美妆,几千块的护肤品，让一起发个礼袋，说了急要送人，还要确认收货才能送，送过来的这个礼袋什么玩意，...,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...


In [55]:
print(df_raw['img_local_path'].isnull().sum())

12


In [66]:
import clip
print(hasattr(clip, "load"))   # 必须是 True

False


In [67]:
import clip
print(clip.__file__)

/opt/conda/envs/py3.11/lib/python3.11/site-packages/clip/__init__.py


In [ ]:
# 截止这里保存一下df_raw

df_raw.to_pickle("df_raw_backup.pkl")

In [2]:
# 重启kernal后
import pandas as pd

df_raw = pd.read_pickle("/mnt/workspace/oss/yyj_ai/fake_reviews_detection/tmp/df_raw_backup.pkl")

In [2]:
import clip
import torch
from PIL import Image
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)

@torch.no_grad()
def preprocess_all_images(df):
    image_tensors = []

    for p in tqdm(df["img_local_path"], desc="CLIP image preprocess"):
        if isinstance(p, str):
            try:
                img = Image.open(p).convert("RGB")
                image_tensors.append(clip_preprocess(img))
            except:
                image_tensors.append(EMPTY_IMAGE)
        else:
            image_tensors.append(EMPTY_IMAGE)

    return image_tensors


100%|███████████████████████████████████████| 338M/338M [04:49<00:00, 1.22MiB/s]


In [3]:
from torch.utils.data import Dataset

class ReviewCLIPDataset(Dataset):
    def __init__(self, df):
        self.texts = df["text_clean"].tolist()
        self.images = df["image_tensor"].tolist()
        self.labels = df["label"].tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "text": self.texts[idx],
            "image": self.images[idx],   # 一定是 Tensor
            "label": int(self.labels[idx])
        }

### 数据集采样和划分

In [3]:
df_raw

,user_id,review_time,image_url,review_text,label,genre,text_clean,img_local_path
0,花***灵,2021-06-08,https://jvod.300hu.com/img/2021/81489379/1/img...,JM家面膜，多年来一直在用。这款水母面膜，适合各种肤质，里面的精华很多，补水效果效果很明显呢...,1,美妆,JM家面膜，多年来一直在用。这款水母面膜，适合各种肤质，里面的精华很多，补水效果效果很明显呢...,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
1,不***头,2022-12-24,https://img30.360buyimg.com/shaidan/s300x300_j...,挑了好久，终于选择这款JM的水库面膜，简直是太棒了，强效水光，一剂深入补水，轻松应急，每一片...,1,美妆,挑了好久，终于选择这款JM的水库面膜，简直是太棒了，强效水光，一剂深入补水，轻松应急，每一片...,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
2,新***络,2022-07-20,https://img30.360buyimg.com/shaidan/s300x300_j...,一直在用，JM 的水库面膜，简直是太棒了，双十一活动，我就多屯了一些朋友和我都觉得特别好用，...,1,美妆,一直在用，JM 的水库面膜，简直是太棒了，双十一活动，我就多屯了一些朋友和我都觉得特别好用，...,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
3,L***家,2023-07-21,https://img30.360buyimg.com/shaidan/s300x300_j...,一直在用新款面膜，效果很赞！特别是三部曲，三步护理，洗面奶，眼霜，精华，感觉竟然比自己买的某...,1,美妆,一直在用新款面膜，效果很赞！特别是三部曲，三步护理，洗面奶，眼霜，精华，感觉竟然比自己买的某...,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
4,j***3,2022-06-17,https://img30.360buyimg.com/shaidan/s300x300_j...,JM 的面膜，简直是太棒了，朋友和我都觉得特别好用， 我觉得各种肌肤都挺实用，尤其是那种长期...,1,美妆,JM 的面膜，简直是太棒了，朋友和我都觉得特别好用， 我觉得各种肌肤都挺实用，尤其是那种长期...,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
...,...,...,...,...,...,...,...,...
7797,匿名买家,2025年6月22日,//img.alicdn.com/imgextra/i3/46116860184273839...,人都要疯了 用完狂爆痘！！！真的要谨慎购买,0,美妆,人都要疯了 用完狂爆痘！！！真的要谨慎购买,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
7798,匿名买家,2025年6月21日,//img.alicdn.com/imgextra/i2/46116860184273847...,没有塑封膜，之前买过多次50毫升和15毫升都是有塑封膜，这次没有，怀疑是人家退货的，店铺要求...,0,美妆,没有塑封膜，之前买过多次50毫升和15毫升都是有塑封膜，这次没有，怀疑是人家退货的，店铺要求...,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
7799,微***,2025年6月20日,//img.alicdn.com/imgextra/i2/46116860184273848...,之前一直用兰蔻，这次想换换。就买了黑绷带和sk2的大红瓶面霜试试。官网应该不至于卖假货但收到...,0,美妆,之前一直用兰蔻，这次想换换。就买了黑绷带和sk2的大红瓶面霜试试。官网应该不至于卖假货但收到...,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
7800,匿名买家,2025年6月20日,//img.alicdn.com/imgextra/i4/46116860184273824...,几千块的护肤品，让一起发个礼袋，说了急要送人，还要确认收货才能送，送过来的这个礼袋什么玩意，...,0,美妆,几千块的护肤品，让一起发个礼袋，说了急要送人，还要确认收货才能送，送过来的这个礼袋什么玩意，...,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...


#### 基础清洗 + 分层划分

In [9]:
import pandas as pd
test_df=pd.read_csv(r"D:\②毕设\FRD-CLIP完整代码\tmp_input\test_ready.csv",encoding='utf-8')
test_df['label'].value_counts()

label
0    629
1    540
Name: count, dtype: int64

In [10]:
import pandas as pd
train_df=pd.read_csv(r"D:\②毕设\FRD-CLIP完整代码\tmp_input\train_ready.csv",encoding='utf-8')
train_df['label'].value_counts()

label
0    2933
1    2520
Name: count, dtype: int64

In [11]:
import pandas as pd
val_df=pd.read_csv(r"D:\②毕设\FRD-CLIP完整代码\tmp_input\val_ready.csv",encoding='utf-8')
val_df['label'].value_counts()

label
0    628
1    540
Name: count, dtype: int64

In [4]:
from sklearn.model_selection import train_test_split

# ===== 1. 拷贝原始清洗后的数据 =====
df_all = df_raw.copy()

# 可选：丢弃极端无效样本（建议保留）
df_all = df_all[df_all["text_clean"].notna()].reset_index(drop=True)
df_all = df_all[df_all["img_local_path"].notna()].reset_index(drop=True)

# ===== 2. 分层划分（防止类别比例偏移）=====
df_train, df_temp = train_test_split(
    df_all,
    test_size=0.3,
    stratify=df_all["label"],
    random_state=42
)

df_val, df_test = train_test_split(
    df_temp,
    test_size=0.5,
    stratify=df_temp["label"],
    random_state=42
)

print(f"训练集: {len(df_train)}")
print(f"验证集: {len(df_val)}")
print(f"测试集: {len(df_test)}")

print("\n各集合标签分布:")
for name, subset in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    print(f"{name}: {subset['label'].value_counts().to_dict()}")


训练集: 5453
验证集: 1168
测试集: 1169

各集合标签分布:
Train: {0: 2933, 1: 2520}
Val: {0: 628, 1: 540}
Test: {0: 629, 1: 540}


#### 仅对训练集构造加权采样器

类别不平衡问题仅在训练阶段通过加权采样缓解，验证与测试集保持真实分布。

In [5]:
import torch
from torch.utils.data import WeightedRandomSampler

# ===== 3. 仅使用训练集计算类别权重 =====
class_counts = df_train['label'].value_counts().sort_index().values
class_weights = 1.0 / class_counts

# 为每个训练样本分配权重
sample_weights = torch.tensor(
    [class_weights[label] for label in df_train['label']],
    dtype=torch.double
)

# 加权随机采样器（仅用于训练）
weighted_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

print(f"类别权重: {dict(enumerate(class_weights))}")


类别权重: {0: np.float64(0.00034094783498124785), 1: np.float64(0.0003968253968253968)}


#### 多模态 Dataset

In [6]:
from torch.utils.data import Dataset

class MultimodalReviewDataset(Dataset):
    def __init__(self, dataframe, processor, image_transform=None, phase='train'):
        """
        Args:
            dataframe: 包含 text_clean / img_local_path / label 的 DataFrame
            processor: CLIP 文本处理器
            image_transform: 图像预处理（ResNet / CLIP）
            phase: train / val / test（用于控制是否启用增强）
        """
        self.df = dataframe.reset_index(drop=True)
        self.processor = processor
        self.image_transform = image_transform
        self.phase = phase

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # ===== 文本处理（CLIP 限制 max_length=77）=====
        text_inputs = self.processor(
            text=row["text_clean"],
            truncation=True,
            max_length=77,
            return_tensors="pt"
        )
        text_tensor = text_inputs["input_ids"].squeeze(0)

        # ===== 图像处理 =====
        img_tensor = load_and_transform_image(
            row["img_local_path"],
            transform=self.image_transform
        )

        label = torch.tensor(row["label"], dtype=torch.long)

        return {
            "text": text_tensor,
            "image": img_tensor,
            "label": label
        }


#### DataLoader

In [10]:
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained(
    "/mnt/workspace/oss/yyj_ai/fake_reviews_detection/openai/clip-vit-base-patch32/"
)

In [11]:
print(processor)

CLIPProcessor:
- image_processor: CLIPImageProcessor {
  "crop_size": {
    "height": 224,
    "width": 224
  },
  "do_center_crop": true,
  "do_convert_rgb": true,
  "do_normalize": true,
  "do_rescale": true,
  "do_resize": true,
  "image_mean": [
    0.48145466,
    0.4578275,
    0.40821073
  ],
  "image_processor_type": "CLIPImageProcessor",
  "image_std": [
    0.26862954,
    0.26130258,
    0.27577711
  ],
  "resample": 3,
  "rescale_factor": 0.00392156862745098,
  "size": {
    "shortest_edge": 224
  }
}

- tokenizer: CLIPTokenizerFast(name_or_path='/mnt/workspace/oss/yyj_ai/fake_reviews_detection/openai/clip-vit-base-patch32/', vocab_size=49408, model_max_length=77, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|startoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	49406: AddedToken("<|startoftext|>", rstrip=False, lstr

In [13]:
image_transform = processor.image_processor

from PIL import Image

def load_and_transform_image(img_path, transform):
    image = Image.open(img_path).convert("RGB")
    image = transform(images=image, return_tensors="pt")["pixel_values"]
    return image.squeeze(0)

In [14]:
from torch.utils.data import DataLoader

# ===== 4. 构建数据集 =====
train_dataset = MultimodalReviewDataset(df_train, processor, image_transform, phase='train')
val_dataset   = MultimodalReviewDataset(df_val, processor, image_transform, phase='val')
test_dataset  = MultimodalReviewDataset(df_test, processor, image_transform, phase='test')

# ===== 5. DataLoader =====
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    sampler=weighted_sampler,   # 仅训练集使用采样器
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")


Train batches: 171
Val batches: 37
Test batches: 37


In [15]:
df_all

,user_id,review_time,image_url,review_text,label,genre,text_clean,img_local_path
0,花***灵,2021-06-08,https://jvod.300hu.com/img/2021/81489379/1/img...,JM家面膜，多年来一直在用。这款水母面膜，适合各种肤质，里面的精华很多，补水效果效果很明显呢...,1,美妆,JM家面膜，多年来一直在用。这款水母面膜，适合各种肤质，里面的精华很多，补水效果效果很明显呢...,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
1,不***头,2022-12-24,https://img30.360buyimg.com/shaidan/s300x300_j...,挑了好久，终于选择这款JM的水库面膜，简直是太棒了，强效水光，一剂深入补水，轻松应急，每一片...,1,美妆,挑了好久，终于选择这款JM的水库面膜，简直是太棒了，强效水光，一剂深入补水，轻松应急，每一片...,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
2,新***络,2022-07-20,https://img30.360buyimg.com/shaidan/s300x300_j...,一直在用，JM 的水库面膜，简直是太棒了，双十一活动，我就多屯了一些朋友和我都觉得特别好用，...,1,美妆,一直在用，JM 的水库面膜，简直是太棒了，双十一活动，我就多屯了一些朋友和我都觉得特别好用，...,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
3,L***家,2023-07-21,https://img30.360buyimg.com/shaidan/s300x300_j...,一直在用新款面膜，效果很赞！特别是三部曲，三步护理，洗面奶，眼霜，精华，感觉竟然比自己买的某...,1,美妆,一直在用新款面膜，效果很赞！特别是三部曲，三步护理，洗面奶，眼霜，精华，感觉竟然比自己买的某...,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
4,j***3,2022-06-17,https://img30.360buyimg.com/shaidan/s300x300_j...,JM 的面膜，简直是太棒了，朋友和我都觉得特别好用， 我觉得各种肌肤都挺实用，尤其是那种长期...,1,美妆,JM 的面膜，简直是太棒了，朋友和我都觉得特别好用， 我觉得各种肌肤都挺实用，尤其是那种长期...,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
...,...,...,...,...,...,...,...,...
7785,匿名买家,2025年6月22日,//img.alicdn.com/imgextra/i3/46116860184273839...,人都要疯了 用完狂爆痘！！！真的要谨慎购买,0,美妆,人都要疯了 用完狂爆痘！！！真的要谨慎购买,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
7786,匿名买家,2025年6月21日,//img.alicdn.com/imgextra/i2/46116860184273847...,没有塑封膜，之前买过多次50毫升和15毫升都是有塑封膜，这次没有，怀疑是人家退货的，店铺要求...,0,美妆,没有塑封膜，之前买过多次50毫升和15毫升都是有塑封膜，这次没有，怀疑是人家退货的，店铺要求...,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
7787,微***,2025年6月20日,//img.alicdn.com/imgextra/i2/46116860184273848...,之前一直用兰蔻，这次想换换。就买了黑绷带和sk2的大红瓶面霜试试。官网应该不至于卖假货但收到...,0,美妆,之前一直用兰蔻，这次想换换。就买了黑绷带和sk2的大红瓶面霜试试。官网应该不至于卖假货但收到...,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
7788,匿名买家,2025年6月20日,//img.alicdn.com/imgextra/i4/46116860184273824...,几千块的护肤品，让一起发个礼袋，说了急要送人，还要确认收货才能送，送过来的这个礼袋什么玩意，...,0,美妆,几千块的护肤品，让一起发个礼袋，说了急要送人，还要确认收货才能送，送过来的这个礼袋什么玩意，...,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...


In [19]:
df_train.to_pickle("/mnt/workspace/oss/yyj_ai/fake_reviews_detection/tmp/train.pkl")
df_val.to_pickle("/mnt/workspace/oss/yyj_ai/fake_reviews_detection/tmp/val.pkl")
df_test.to_pickle("/mnt/workspace/oss/yyj_ai/fake_reviews_detection/tmp/test.pkl")
df_all.to_pickle("/mnt/workspace/oss/yyj_ai/fake_reviews_detection/tmp/all_cleaned_backup.pkl")

In [20]:
df_train

,user_id,review_time,image_url,review_text,label,genre,text_clean,img_local_path
7010,m**3,2025年11月20日,//img.alicdn.com/imgextra/i2/46116860184273849...,整体评价： 保湿控油效果：好，质量好，服務各方面都好，值得购买,0,美妆,整体评价 保湿控油效果 好，质量好，服务各方面都好，值得购买,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
519,i**?,2025年10月31日,//img.alicdn.com/imgextra/i3/46116860184273802...,常年买这款钙片，孕期吃，家里老人也在吃，迷你钙比较好吞服！补充钙的同时也补维生素D，促进吸收...,1,保健,常年买这款钙片，孕期吃，家里老人也在吃，迷你钙比较好吞服！补充钙的同时也补维生素D，促进吸收...,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
6674,一**火,2025年11月12日,//img.alicdn.com/imgextra/i4/46116860184273878...,喝酒熬夜必备，产品不错，性价比高，第二次回购了,0,保健,喝酒熬夜必备，产品不错，性价比高，第二次回购了,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
6936,公**9,2025年10月30日,//img.alicdn.com/imgextra/i3/46116860184273829...,霜很水润，给婆婆买的，一直用这款还是一如既往的好用,0,美妆,霜很水润，给婆婆买的，一直用这款还是一如既往的好用,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
550,瑟***,2025年9月14日,//img.alicdn.com/imgextra/i3/46116860184273864...,用玉兰油已经多年，然，仍然觉得她是性价比最高的基础保湿面霜，敏感肌在换季时维稳也很躜,1,美妆,用玉兰油已经多年，然，仍然觉得她是性价比最高的基础保湿面霜，敏感肌在换季时维稳也很躜,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
...,...,...,...,...,...,...,...,...
6928,匿名买家,2025年11月25日,//img.alicdn.com/imgextra/i3/46116860184273858...,好用，爱用，每天都在用，特别时候秋东季节防干燥,0,美妆,好用，爱用，每天都在用，特别时候秋东季节防干燥,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
1820,爱**圆,2024年1月1日,//img.alicdn.com/imgextra/i3/0/O1CN01Qsh4Em1JC...,被种草之后立马下单了这款产品，客服真的挺给力的👍，整体质感比预期好，包装打开那一刻就很加分✨...,1,保健,被种草之后立马下单了这款产品，客服真的挺给力的 ，整体质感比预期好，包装打开那一刻就很加分 ...,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
4611,神**鱼,2025年5月6日,//img.alicdn.com/imgextra/i1/46116860184273836...,日期很新 包装完整 物流也是显示过关产品 正品 放心购买,0,保健,日期很新 包装完整 物流也是显示过关产品 正品 放心购买,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
4930,木**花,2025年3月9日,//img.alicdn.com/imgextra/i1/46116860184273804...,Swisse胆碱片不愧是大品牌！成分天然纯净，吃着特别安心。我坚持吃了几个月，记忆力提升明显...,0,保健,Swisse胆碱片不愧是大品牌！成分天然纯净，吃着特别安心。我坚持吃了几个月，记忆力提升明显...,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
